## Imports

In [1]:
import torch
from torchvision.transforms import transforms
import torchvision
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Dataset, random_split, Subset
import kagglehub
from pathlib import Path
import shutil
from PIL import Image
import os
from torch import nn
import optuna

/home/orlandojunior/miniconda3/envs/huggingface/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Leitura e Separação das pastas

In [2]:
# Download latest version
path = kagglehub.dataset_download("sid4sal/alpaca-dataset-small")

print("Path to dataset files:", path)

Path to dataset files: /home/orlandojunior/.cache/kagglehub/datasets/sid4sal/alpaca-dataset-small/versions/1


In [3]:
pasta_destino_alpaca = Path('./alpaca')
pasta_destino_alpaca.mkdir(parents=True, exist_ok=True)

pasta_destino_not_alpaca = Path('./not alpaca/')
pasta_destino_not_alpaca.mkdir(parents=True, exist_ok=True)

caminho_origem = Path(path)
for file in caminho_origem.glob('**/alpaca/*'):
    if file.suffix.lower() in ['.jpg','.jpeg','.png']:
        shutil.copy(file,pasta_destino_alpaca/file.name)
        
for file in caminho_origem.glob('**/not alpaca/*'):
    if file.suffix.lower() in ['.jpg','.jpeg','.png']:
        shutil.copy(file,pasta_destino_not_alpaca/file.name)
print("Imagens salvas com sucesso")

Imagens salvas com sucesso


In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using device: MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Using device: CUDA


## Transformações

In [5]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


## Classes Dataset

In [6]:
class AlpacaDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.classes = ["alpaca", "not alpaca"]
        self.classes_to_idx = {c: i for i,c in enumerate(self.classes)}

        self.samples = []
        
        for class_name in self.classes:
            for img_path in (self.root_dir/class_name).iterdir():
                if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    self.samples.append((img_path,self.classes_to_idx[class_name]))
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [7]:
class SubsetWithTransform(Dataset):

    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        
        return image, label

## Divisão de treino validação e teste

In [8]:
def split_dataset(dataset, val_fraction=0.15, test_fraction=0.15):
    total_size = len(dataset)
    
    val_size = int(total_size * val_fraction)
    test_size = int(total_size * test_fraction)
    train_size = total_size - val_size - test_size
    
    train_dataset, val_dataset, test_dataset = random_split(
        dataset,[train_size,val_size,test_size]
    )
    
    return train_dataset, val_dataset, test_dataset

In [9]:
dataset_raw = AlpacaDataset(Path('./dataset/'),transform=None)
train_subset, val_subset, test_subset = split_dataset(dataset_raw)

In [10]:
train_dataset = SubsetWithTransform(train_subset, transform=train_transforms)
test_dataset = SubsetWithTransform(test_subset, transform=test_transforms)
val_dataset = SubsetWithTransform(val_subset, transform=val_transforms)

print(f"Length of training dataset:   {len(train_dataset)}")
print(f"Length of validation dataset: {len(val_dataset)}")
print(f"Length of test dataset:       {len(test_dataset)}")

Length of training dataset:   229
Length of validation dataset: 49
Length of test dataset:       49


## Model Arctecture

In [11]:
class FlexibleCNN(nn.Module):
    
    def __init__(self, n_layers, n_filters, kernel_sizes, dropout_rate,fc_size):
        super(FlexibleCNN, self).__init__()
        
        blocks = []
        
        in_channels = 3
        
        for i in range(n_layers):
            out_channels = n_filters[i]
            kernel_size = kernel_sizes[i]
            
            padding = (kernel_size - 1) // 2
            
            block = nn.Sequential(
                nn.Conv2d(in_channels=in_channels,out_channels=out_channels,kernel_size=kernel_size, padding=padding),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2,stride=2)
            )
            
            blocks.append(block)
            in_channels = out_channels
        
        self.features = nn.Sequential(*blocks)
        self.dropout_rate = dropout_rate
        self.fc_size = fc_size
        
        self.classifier = None
        
    def _create_classifier(self, flattened_size, device):
        self.classifier = nn.Sequential(
            nn.Dropout(self.dropout_rate),
            nn.Linear(flattened_size, self.fc_size),
            nn.ReLU(inplace=True),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.fc_size,2)
        ).to(device)
        
    def forward(self,x):
        device = x.device
        
        x = self.features(x)
        flattened = torch.flatten(x,1)
        flattened_size = flattened.size(1)
        
        if self.classifier is None:
            self._create_classifier(flattened_size, device)
        return self.classifier(flattened)
        

## Treino e avaliação

In [12]:
def train_epoch(model, train_dataloader, optimizer, loss_fcn, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in tqdm(train_dataloader, desc="Batches", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()              # zera os gradientes do passo anterior
        outputs = model(inputs)            # forward
        loss = loss_fcn(outputs, labels)   # calcula a loss
        loss.backward()                    # backward (gradientes)
        optimizer.step()                   # atualiza os pesos

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    epoch_loss = running_loss / total
    epoch_acc = correct/total

    return epoch_loss, epoch_acc


def train_and_evaluate(model, optimizer, scheduler, device, n_epochs, train_loader, val_loader, patience=None):
    model = model.to(device)
    loss_fcn = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state = None                      # pesos da melhor época
    epochs_without_improvement = 0

    for epoch in tqdm(range(n_epochs), desc="Épocas"):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fcn, device)
        val_acc = evaluate_accuracy(model, val_loader, device)

        print(f"Epoch {epoch+1} - Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc:.4f}")

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_acc)
            else:
                scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if patience is not None and epochs_without_improvement >= patience:
            print(f"Early stopping na época {epoch+1} (sem melhora há {patience} épocas)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"Training complete! Melhor Val Acc: {best_val_acc:.4f}")
    return best_val_acc



def evaluate_accuracy(model, data_loader, device):
    model.eval()                           # desliga dropout, batchnorm em modo de inferência
    total_correct, total_samples = 0, 0

    with torch.no_grad():                  # sem gradientes, economiza memória
        for inputs, labels in tqdm(data_loader, desc="Avaliando", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    return total_correct / total_samples

## Optuna

In [13]:
def objective(trial, device):
    
    n_layers = trial.suggest_int("n_layers", 2,5)
    n_filters = [trial.suggest_int(f"n_filters_{i}", 16,128) for i in range(n_layers)]
    kernel_sizes = [trial.suggest_categorical(f"kernel_size_{i}", [3,5]) for i in range(n_layers)]
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    fc_size = trial.suggest_int("fc_size", 64, 256)
    
    model = FlexibleCNN(n_layers, n_filters, kernel_sizes, dropout_rate, fc_size).to(device)
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    model(dummy_input)
    
    learning_rate = 0.001
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    
    batch_size = 128
    train_dataloader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    val_dataloader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False)
    
    n_epochs = 10
    
    accuracy = train_and_evaluate(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        n_epochs=n_epochs,
        train_loader=train_dataloader,
        val_loader=val_dataloader
    )
    
    return accuracy


In [14]:
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: objective(trial, device), n_trials=20)


[I 2026-09-27 21:05:51,617] A new study created in memory with name: no-name-d2825059-4ed9-4646-beca-c80896cb141c
Épocas:  10%|█         | 1/10 [00:03<00:28,  3.18s/it]

Epoch 1 - Train Loss: 2.3977 - Train Acc: 0.5721 - Val Acc: 0.4082


Épocas:  20%|██        | 2/10 [00:06<00:24,  3.01s/it]

Epoch 2 - Train Loss: 0.9872 - Train Acc: 0.4236 - Val Acc: 0.5102


Épocas:  30%|███       | 3/10 [00:09<00:20,  2.99s/it]

...

Epoch 8 - Train Loss: 0.6848 - Train Acc: 0.5764 - Val Acc: 0.5918


Épocas:  90%|█████████ | 9/10 [03:33<00:23, 23.75s/it]

Epoch 9 - Train Loss: 0.6848 - Train Acc: 0.5764 - Val Acc: 0.5918


Épocas: 100%|██████████| 10/10 [03:54<00:00, 23.45s/it]
[I 2026-09-27 21:52:10,580] Trial 19 finished with value: 0.5918367346938775 and parameters: {'n_layers': 2, 'n_filters_0': 57, 'n_filters_1': 96, 'kernel_size_0': 3, 'kernel_size_1': 5, 'dropout_rate': 0.31666368860254324, 'fc_size': 106}. Best is trial 17 with value: 0.673469387755102.


Epoch 10 - Train Loss: 0.6848 - Train Acc: 0.5764 - Val Acc: 0.5918
Training complete! Melhor Val Acc: 0.5918


## Modelo final com os melhores hiperparâmetros

In [15]:
best = study.best_params
n_layers = best["n_layers"]

best_model = FlexibleCNN(
    n_layers=n_layers,
    n_filters=[best[f"n_filters_{i}"] for i in range(n_layers)],
    kernel_sizes=[best[f"kernel_size_{i}"] for i in range(n_layers)],
    dropout_rate=best["dropout_rate"],
    fc_size=best["fc_size"],
).to(device)

# o classificador é criado no primeiro forward, então rode o dummy ANTES de criar o optimizer
best_model(torch.randn(1, 3, 224, 224).to(device))

optimizer = torch.optim.Adam(best_model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

train_and_evaluate(best_model, optimizer, scheduler, device, 50, train_loader, val_loader, patience=15)
print(f"Test accuracy: {evaluate_accuracy(best_model, test_loader, device):.4f}")

Épocas:   2%|▏         | 1/50 [00:22<18:18, 22.41s/it]

Epoch 1 - Train Loss: 23.1458 - Train Acc: 0.5721 - Val Acc: 0.5918


Épocas:   4%|▍         | 2/50 [00:45<18:23, 22.99s/it]

Epoch 2 - Train Loss: 0.8372 - Train Acc: 0.5197 - Val Acc: 0.5714


Épocas:   6%|▌         | 3/50 [01:13<19:35, 25.02s/it]

...

Épocas:  30%|███       | 15/50 [06:12<13:36, 23.32s/it]

Epoch 15 - Train Loss: 0.6968 - Train Acc: 0.4236 - Val Acc: 0.4082


Épocas:  30%|███       | 15/50 [06:34<15:21, 26.33s/it]


Epoch 16 - Train Loss: 0.6986 - Train Acc: 0.4236 - Val Acc: 0.4082
Early stopping na época 16 (sem melhora há 15 épocas)
Training complete! Melhor Val Acc: 0.5918


Test accuracy: 0.4898
